# SQL-Übung: WHERE oder HAVING?

6 SQL-Aufgaben zur Datenbank aus AP 1 (`smartphone_dealer_data_backup.db`). Bei jeder Aufgabe müssen Sie selbst entscheiden: **nur WHERE, nur HAVING oder beides?**

Die Kontrollzeile («Ergebnis: … Zeilen») hilft Ihnen beim Überprüfen. Legen Sie die Datenbank in denselben Ordner wie dieses Notebook.

## Merkregel: WHERE oder HAVING?

<table>
<tr><th></th><th>WHERE</th><th>HAVING</th></tr>
<tr><td><b>filtert …</b></td><td>einzelne <b>Zeilen</b></td><td>ganze <b>Gruppen</b></td></tr>
<tr><td><b>wann?</b></td><td><b>vor</b> dem <code>GROUP BY</code></td><td><b>nach</b> dem <code>GROUP BY</code></td></tr>
<tr><td><b>Bedingung auf …</b></td><td>Spaltenwerte: <code>price &gt; 1000</code>, <code>status = 'Sold'</code>, Datum</td><td>Aggregate: <code>COUNT(*) &gt;= 3</code>, <code>SUM(...) &gt; 85000</code>, <code>AVG(...) &gt; 180</code></td></tr>
</table>

**Testfrage:** *Kann ich die Bedingung an einer einzelnen Zeile prüfen, ohne die anderen Zeilen zu kennen?*
- **Ja** → `WHERE` (z.B. «Ist dieses Telefon teurer als 1000?»)
- **Nein**, ich muss zuerst zählen, summieren oder mitteln → `HAVING` (z.B. «Hat diese Marke mindestens 1000 Telefone?»)

**Ausführungsreihenfolge:** `FROM` → `JOIN` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT` → `ORDER BY` → `LIMIT`

Typische Fehler:
- `WHERE COUNT(*) > 3` → Fehler (*misuse of aggregate*), weil `WHERE` vor dem Gruppieren läuft.
- `HAVING price > 1000` für eine Zeilenbedingung → falscher Ort; gehört in `WHERE`.
- In PostgreSQL darf im `HAVING` **kein Alias** aus dem `SELECT` stehen (`HAVING anzahl >= 3` geht nicht) → das Aggregat ausschreiben: `HAVING COUNT(*) >= 3`.
- Viele Aufgaben brauchen **beides**: `WHERE` filtert zuerst die Zeilen, `HAVING` danach die Gruppen.

## Setup

In [1]:
# Libraries
import pandas as pd
from sqlalchemy import create_engine

# Show all columns
pd.set_option('display.max_columns', None)

---
#### <b>Aufgabe (1): Umsatz pro Verkäufer im Jahr 2023</b>

Wie viele Smartphones hat jeder Verkäufer **im Jahr 2023** verkauft, und welchen Umsatz hat er damit erzielt?

<b>Details zur Aufgabenstellung:</b>
- Tabellen: `sales`, `salespersons`
- Spalten: firstname, lastname, anzahl_verkaeufe, umsatz (auf 2 Dezimalstellen gerundet)
- Sortierung: absteigend nach Umsatz

*Kontrolle: Ergebnis hat 5 Zeilen.*

**WHERE, HAVING oder beides? Begründung:**

> [Hier eintragen]

In [2]:
# Engine für Datenbankverbindung erstellen
engine = create_engine('sqlite:///smartphone_dealer_data_backup.db')

# Abfrage erstellen und Ergebnis in einem DataFrame speichern
df_1 = pd.read_sql_query("""
SELECT firstname, lastname, COUNT(saleID) as anzahl_verkäufe, ROUND(CAST(SUM(s.saleprice) AS NUMERIC), 2) AS umsatz

FROM sales s
JOIN salespersons sp ON sp.salespersonid = s.salespersonid
WHERE s.saledate >= '2023-01-01' AND s.saledate < '2024-01-01'
GROUP BY firstname, lastname
ORDER BY umsatz DESC

""", con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_1.head(10)

,firstname,lastname,anzahl_verkäufe,umsatz
0,Charlie,Patel,125,90053.23
1,Edward,Evans,125,88327.17
2,Alice,Nguyen,124,87792.08
3,Diana,Morris,125,86101.69
4,Bob,Kim,125,84956.81


---
#### <b>Aufgabe (2): Marken mit grossem Bestand</b>

Welche Marken haben **mindestens 1000 Telefone** in der Tabelle `phones`? Geben Sie die Anzahl und den Durchschnittspreis aus.

<b>Details zur Aufgabenstellung:</b>
- Tabelle: `phones`
- Spalten: brand, anzahl_telefone, durchschnittspreis (2 Dezimalstellen)
- Sortierung: absteigend nach Anzahl

*Kontrolle: Ergebnis hat 6 Zeilen.*

**WHERE, HAVING oder beides? Begründung:**

> [Hier eintragen]

In [3]:
# Engine für Datenbankverbindung erstellen
engine = create_engine('sqlite:///smartphone_dealer_data_backup.db')

# Abfrage erstellen und Ergebnis in einem DataFrame speichern
df_2 = pd.read_sql_query("""
SELECT brand, COUNT(phoneid) AS anzahl_telefone, ROUND(AVG(price),2) AS durchschnittspreis FROM phones
GROUP BY brand
HAVING COUNT(phoneid) >= 1000
ORDER BY anzahl_telefone DESC
""", con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_2

,brand,anzahl_telefone,durchschnittspreis
0,Oppo,1037,848.35
1,Nokia,1037,845.85
2,Samsung,1013,848.52
3,Xiaomi,1005,837.45
4,LG,1005,856.51
5,OnePlus,1000,855.89


---
#### <b>Aufgabe (3): Stammkunden</b>

Welche Kunden haben **mindestens 3 Smartphones** gekauft? Geben Sie die Anzahl Käufe und die gesamten Ausgaben pro Kunde aus.

<b>Details zur Aufgabenstellung:</b>
- Tabellen: `customers`, `sales`
- Spalten: customerid, firstname, lastname, anzahl_kaeufe, total_ausgaben (2 Dezimalstellen)
- Sortierung: absteigend nach Anzahl Käufe, dann absteigend nach Ausgaben
- Achtung: Mehrere Kunden können denselben Vor- und Nachnamen haben.

*Kontrolle: Ergebnis hat 34 Zeilen.*

**WHERE, HAVING oder beides? Begründung:**

> [Hier eintragen]

In [4]:
# Engine für Datenbankverbindung erstellen
engine = create_engine('sqlite:///smartphone_dealer_data_backup.db')

# Abfrage erstellen und Ergebnis in einem DataFrame speichern
df_3 = pd.read_sql_query("""
SELECT c.customerid, c.firstname, c.lastname, COUNT(c.customerid) AS anzahl_kaufe, ROUND(AVG(s.saleprice),2) AS total_ausgaben
FROM customers c
JOIN sales AS s ON s.customerid= c.customerid
GROUP BY c.customerid,c.firstname, c.lastname
HAVING COUNT(s.saleid) >= 3
ORDER BY anzahl_kaufe DESC, total_ausgaben  DESC
""", con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_3

,customerid,firstname,lastname,anzahl_kaufe,total_ausgaben
0,CU0BLHKHB0LP,John,Johnson,5,654.15
1,CUXBXAHY2H5B,David,Davis,4,726.79
2,CUVY2YTEFOMC,David,Miller,3,925.33
3,CUV20072SJBV,Robert,Rodriguez,3,813.05
4,CUX8RGXNJQRB,Ashley,Johnson,3,796.42
5,CU1SBQ0LJ7T4,David,Garcia,3,788.21
6,CUT7LTC6Y11V,Michael,Brown,3,759.35
7,CUTVHVVZA3SH,Jessica,Miller,3,740.83
8,CUJ101OD21C6,Robert,Martinez,3,737.54
9,CUXN77TS2XHR,Emily,Miller,3,712.84


---
#### <b>Aufgabe (4): Verfügbare Premium-Telefone pro Marke</b>

Wie viele Telefone **über 1000 USD** sind pro Marke **aktuell verfügbar** (`status = 'Available'`)?

<b>Details zur Aufgabenstellung:</b>
- Tabelle: `phones`
- Spalten: brand, anzahl_premium_verfuegbar
- Sortierung: absteigend nach Anzahl

*Kontrolle: Ergebnis hat 10 Zeilen.*

**WHERE, HAVING oder beides? Begründung:**

> [Hier eintragen]

In [5]:
# Engine für Datenbankverbindung erstellen
engine = create_engine('sqlite:///smartphone_dealer_data_backup.db')

# Abfrage erstellen und Ergebnis in einem DataFrame speichern
df_4 = pd.read_sql_query("""
SELECT brand, COUNT(*) AS anzahl_premium_verfügbar FROM phones
WHERE status = 'Available' AND price > 1000
GROUP BY brand
ORDER BY anzahl_premium_verfügbar DESC

""", con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_4

,brand,anzahl_premium_verfügbar
0,OnePlus,111
1,Nokia,105
2,Samsung,103
3,Sony,98
4,Google,95
5,Apple,95
6,Oppo,92
7,Huawei,92
8,Xiaomi,85
9,LG,83


---
#### <b>Aufgabe (5): Teure Techniker</b>

Welche Techniker haben bei ihren **Services ab dem 01.01.2022** **durchschnittliche Kosten von über 180 USD**?

<b>Details zur Aufgabenstellung:</b>
- Tabellen: `servicerecords`, `technicians`
- Spalten: technicianid, firstname, lastname, anzahl_services, durchschnittskosten (2 Dezimalstellen)
- Sortierung: absteigend nach Durchschnittskosten
- Achtung: Es gibt Techniker mit gleichem Namen.

*Kontrolle: Ergebnis hat 4 Zeilen.*

**WHERE, HAVING oder beides? Begründung:**

> [Hier eintragen]

In [6]:
# Engine für Datenbankverbindung erstellen
engine = create_engine('sqlite:///smartphone_dealer_data_backup.db')

# Abfrage erstellen und Ergebnis in einem DataFrame speichern
df_5 = pd.read_sql_query("""
SELECT t.technicianid, 
t.firstname, 
t.lastname, 
COUNT(s.serviceid) AS anzahl_services, 
ROUND(CAST(AVG(s.cost) AS NUMERIC), 2) AS durchschnittskosten 
FROM technicians t 
JOIN servicerecords as s on t.technicianid = s.technicianid
WHERE servicedate >= '2022-01-01'
GROUP BY t.technicianid, t.firstname, t.lastname
HAVING durchschnittskosten > 180
ORDER BY durchschnittskosten DESC;


""", con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_5.head(10)

,technicianid,firstname,lastname,anzahl_services,durchschnittskosten
0,TCDYR2OC3WG,Ivan,Lee,49,188.77
1,T39U51MA5TO,Mia,Lee,30,186.58
2,TNR9S2V7KFY,Olivia,Santos,30,183.57
3,T3LEOBDOCQ6,Liam,Santos,31,182.47


---
#### <b>Aufgabe (6): Starke Verkaufsjahre</b>

Wie hoch war der Umsatz **pro Verkäufer und Jahr** in den Jahren **2022 und 2023**? Zeigen Sie nur Kombinationen aus Verkäufer und Jahr mit einem **Umsatz über 85'000 USD**.

<b>Details zur Aufgabenstellung:</b>
- Tabellen: `sales`, `salespersons`
- Spalten: firstname, lastname, jahr, anzahl_verkaeufe, umsatz (2 Dezimalstellen)
- Sortierung: absteigend nach Umsatz
- Tipp: Das Jahr erhalten Sie mit `SUBSTR(s.saledate, 1, 4)`. Gruppieren Sie nach Verkäufer **und** Jahr.

*Kontrolle: Ergebnis hat 4 Zeilen.*

**WHERE, HAVING oder beides? Begründung:**

> [Hier eintragen]

In [7]:
# Engine für Datenbankverbindung erstellen
engine = create_engine('sqlite:///smartphone_dealer_data_backup.db')

# Abfrage erstellen und Ergebnis in einem DataFrame speichern
df_6 = pd.read_sql_query("""
SELECT sp.firstname, sp.lastname,
       SUBSTR(s.saledate, 1, 4) AS jahr,
       COUNT(s.saleid) AS anzahl_verkaeufe,
       ROUND(CAST(SUM(s.saleprice) AS NUMERIC), 2) AS umsatz
FROM sales AS s
JOIN salespersons AS sp ON s.salespersonid = sp.salespersonid
WHERE s.saledate >= '2022-01-01'
GROUP BY sp.salespersonid, sp.firstname, sp.lastname, SUBSTR(s.saledate, 1, 4)
HAVING SUM(s.saleprice) > 85000
ORDER BY umsatz DESC;
""", con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_6.head(10)

,firstname,lastname,jahr,anzahl_verkaeufe,umsatz
0,Charlie,Patel,2023,125,90053.23
1,Edward,Evans,2023,125,88327.17
2,Alice,Nguyen,2023,124,87792.08
3,Diana,Morris,2023,125,86101.69
